# 🌿 FloraSense AI — Indoor Plant Transfer Learning (GPU Training)

Train **EfficientNet-B0** (primary) and **MobileNetV3-Small** (lightweight) on 20 indoor plant species  
using the **House Plant Species dataset** (kakasher/house-plant-species, CC BY 3.0, ~14,790 images).

> ⚠️ **Dataset note:** Images were collected from Bing Images and curated by a non-expert.
> Verify label quality before relying on model outputs in production.

### Pipeline:
1. GPU verification & dependency setup  
2. Clone FloraSense AI repository  
3. Download & prepare the House Plant Species dataset  
4. Class coverage verification  
5. EfficientNet-B0 transfer learning (head warmup → backbone fine-tuning)  
6. MobileNetV3-Small training for comparison  
7. OOD threshold calibration on validation set  
8. Evaluation on held-out test set (genuine images only)  
9. Download checkpoint for local use

In [ ]:
# ── 1. GPU Check ─────────────────────────────────────────────────────────────
import subprocess, torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:             {torch.cuda.get_device_name(0)}")
    print(f"VRAM:            {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected. Go to Runtime > Change runtime type > GPU")

In [ ]:
# ── 2. Install dependencies ───────────────────────────────────────────────────
!pip install -q torch torchvision pillow scikit-learn matplotlib seaborn

In [ ]:
# ── 3. Clone FloraSense AI repository ────────────────────────────────────────
# Replace with your actual repository URL
REPO_URL = "https://github.com/YOUR_USERNAME/FloraSense-AI.git"  # ← Update this

import os
if not os.path.exists("/content/FloraSense-AI"):
    !git clone {REPO_URL} /content/FloraSense-AI

%cd /content/FloraSense-AI
!ls -la

In [ ]:
# ── 4. Download & prepare the dataset ────────────────────────────────────────
# This downloads ~150 MB from Hugging Face, extracts 20 indoor species,
# and creates train/val/test splits (70/15/15).
#
# IMPORTANT: kakasher/house-plant-species CC BY 3.0
# Citation: Gregorowicz, K. (2024). House Plant Species Dataset.
# https://huggingface.co/datasets/kakasher/house-plant-species

!python download_real_dataset.py --data_dir ./data --min_images 50

# Show per-class counts
import json, os
if os.path.exists('./data/dataset_manifest.json'):
    with open('./data/dataset_manifest.json') as f:
        manifest = json.load(f)
    print("\n📊 Per-class image counts (total valid images):")
    for cls, count in sorted(manifest['class_counts'].items()):
        status = '✅' if count >= 50 else '⚠️'
        print(f"  {status} {cls:30s}: {count:4d}")
    print(f"\nTotal images saved: {manifest['total_images']}")
    if manifest['missing_classes']:
        print(f"⚠️  Missing classes: {manifest['missing_classes']}")
    if manifest['insufficient_classes']:
        print(f"⚠️  Under-represented: {manifest['insufficient_classes']}")

In [ ]:
# ── 5a. Train EfficientNet-B0 ─────────────────────────────────────────────────
# Phase 1 (5 epochs): freeze backbone, train classification head
# Phase 2 (15 epochs): unfreeze top layers, fine-tune at low LR
# Uses early stopping (patience=5), AdamW, cosine LR schedule

!python train_models.py \
    --arch efficientnet_b0 \
    --data_dir ./data \
    --warmup_epochs 5 \
    --finetune_epochs 15 \
    --batch_size 32 \
    --head_lr 5e-4 \
    --backbone_lr 1e-5 \
    --patience 5 \
    --checkpoint_dir ./checkpoints \
    --seed 42

In [ ]:
# ── 5b. Train MobileNetV3-Small (optional comparison) ────────────────────────
!python train_models.py \
    --arch mobilenet_v3_small \
    --data_dir ./data \
    --warmup_epochs 5 \
    --finetune_epochs 15 \
    --batch_size 32 \
    --head_lr 5e-4 \
    --backbone_lr 1e-5 \
    --patience 5 \
    --checkpoint_dir ./checkpoints \
    --seed 42

In [ ]:
# ── 6. Verify checkpoints ─────────────────────────────────────────────────────
from model import InferenceEngine

for arch in ['efficientnet_b0', 'mobilenet_v3_small']:
    results = InferenceEngine.list_available('./checkpoints')
    for r in results:
        if r['arch'] == arch:
            if r['is_usable']:
                print(f"✅ {arch}: {r['num_classes']} classes, best_val_acc={r['best_acc']:.4f}")
            else:
                print(f"❌ {arch}: {r.get('error', 'not found')}")

In [ ]:
# ── 7. Evaluate on held-out test set ─────────────────────────────────────────
# Only runs if real test images exist in ./data/test/
# Reports genuine Top-1, Top-3, macro-F1, confusion matrix

import os
test_dir = './data/test'
if os.path.isdir(test_dir) and any(os.scandir(test_dir)):
    !python evaluate_models.py --checkpoint_dir ./checkpoints --test_dir ./data/test
else:
    print("⚠️  No test data found. Run download_real_dataset.py first.")

In [ ]:
# ── 8. Show evaluation results ───────────────────────────────────────────────
import json
if os.path.exists('evaluation_report.json'):
    with open('evaluation_report.json') as f:
        report = json.load(f)
    for model_name, metrics in report.items():
        print(f"\n{'='*50}")
        print(f"Model: {model_name}")
        print(f"{'='*50}")
        for k, v in metrics.items():
            if v is not None:
                print(f"  {k}: {v}")
else:
    print("No evaluation_report.json found. Run the evaluation cell first.")

In [ ]:
# ── 9. Display confusion matrix ───────────────────────────────────────────────
from IPython.display import Image as IPImage, display
import os

for fname in ['model_comparison_confusion_matrix.png', 'confusion_matrix.png']:
    if os.path.exists(fname):
        display(IPImage(fname))
        break

In [ ]:
# ── 10. Download checkpoint package ──────────────────────────────────────────
import shutil
from google.colab import files

# Package checkpoints + class mapping + preprocessing config
shutil.make_archive('florasense_checkpoints', 'zip', '.', 'checkpoints')
files.download('florasense_checkpoints.zip')

print("Downloaded florasense_checkpoints.zip")
print("Extract to your local FloraSense-AI/checkpoints/ directory")

## After Training: Local Use

1. Extract `florasense_checkpoints.zip` to `FloraSense-AI/checkpoints/`
2. Run the app locally:
   ```powershell
   python -m streamlit run app.py
   ```

## Dataset Citation

```
Gregorowicz, K. (2024). House Plant Species Dataset.
Hugging Face Datasets. https://huggingface.co/datasets/kakasher/house-plant-species
License: CC BY 3.0
```

> **Important:** Always attribute the dataset when publishing results.
> Do not claim higher accuracy than actually measured on the genuine test set.